# 5.2.3 · Qualidade dos registros — caminhos do Drive preenchidos
## Polígonos inválidos por ano, 2021–2025

**Pronto para o Google Colab:** execute as células na ordem. Os caminhos e os IDs foram conferidos na pasta **Envio/Base**. O processamento usa blocos de 50 mil linhas para não carregar os arquivos completos na memória.

São produzidos dois diagnósticos independentes:
1. **Cerrado:** arquivos `sicor_cerrado_2021.csv` a `sicor_cerrado_2025.csv`, cerca de 480 mil registros no processamento anterior. É a tabela principal para descrever o recorte territorial da pesquisa.
2. **Entrada nacional:** arquivos `sicor_glebas_wkt_2021.csv` a `sicor_glebas_wkt_2025.csv`, cerca de 5,3 milhões de registros. Permite avaliar a etapa anterior à seleção do Cerrado. Pode ser desativada para uma execução mais rápida.

### Por que a coluna GT_GEOMETRIA?
O código **Base01.ipynb** consultado converte `GT_GEOMETRIA` para uma geometria temporária, corrige essa geometria quando necessário e a usa no filtro `within`. Em seguida, exclui a coluna temporária e **mantém o WKT original em GT_GEOMETRIA** nos CSVs selecionados. Por isso, a auditoria dos CSVs anuais do Cerrado examina esse WKT preservado. Essa interpretação pressupõe que os arquivos não tenham sido posteriormente sobrescritos com WKT corrigido.

O relatório histórico `resumo_processamento_sicor_cerrado_2021_2025.csv` registra 2 geometrias encaminhadas à correção em 2022 e 1 em 2023, **no universo nacional, antes do filtro**. Esses totais não são automaticamente os totais de inválidos do Cerrado. Este notebook faz uma nova contagem.

### Critérios
- Uma linha é um registro. `Polygon` e `MultiPolygon` não vazios são avaliados, sem explodir suas partes e sem deduplicação.
- **Percentual principal = 100 × polígonos inválidos / polígonos avaliados.** Também é calculado o percentual sobre todas as linhas.
- WKT ausente, WKT não interpretável, geometria vazia e tipo não poligonal são categorias separadas.
- Não há correção, recorte ou reprojeção. O ano é o do arquivo de origem, não necessariamente o ano da contratação.
- Percentuais são mostrados com **seis casas decimais**, para que poucos casos em bases grandes não apareçam como 0,00%.

Fontes de configuração consultadas: [pasta Base](https://drive.google.com/drive/folders/17kW1DGXIHaBukS1Mq40a-0W904yYYUia), [Base01.ipynb](https://drive.google.com/file/d/1lB7318tNJWuEJlr4r-EbNpPTCixz-ld8/view), [relatório histórico](https://drive.google.com/file/d/10ATHTY0hG64HycIxLt076ccJTPqBF3Co/view).


## 1. Instalar bibliotecas e conectar o Drive

In [1]:
%pip -q install "pandas>=2.2,<3" "shapely>=2.0,<3" "google-api-python-client>=2,<3" "google-auth>=2,<3"
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


## 2. Caminhos já configurados
Se usar a mesma conta e organização de pastas, não precisa editar caminhos. Se um caminho não existir, o notebook tentará baixar o arquivo pelo ID confirmado, com autenticação. Os arquivos de origem não são modificados. A auditoria nacional exige ler aproximadamente 4 GB adicionais.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
ANOS = [2021, 2022, 2023, 2024, 2025]
PASTA_BASE = Path("/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base")
PASTA_RELATORIOS = Path("/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios")
EXECUTAR_NACIONAL = True   # False = somente recorte do Cerrado
TAMANHO_BLOCO = 50_000
COLUNA_WKT = "GT_GEOMETRIA"
SEPARADOR = ";"
CODIFICACAO = "utf-8-sig"
BAIXAR_ZIP = True
USAR_DOWNLOAD_POR_ID_SE_NECESSARIO = True

IDS_CERRADO = {
    2021: "1mf9QHkD-hj3OKweHqBErGy_BcGa9xcDP",
    2022: "1lo1jYNLdNvxpMeaUPvSRKnHOIjkleiyQ",
    2023: "1cjevMXAB2AXiSKiABx7W0qHmLiOhliGK",
    2024: "16_5WVO3Ft_ESz2O1eBeRo2XFXGoBsd1W",
    2025: "1cuWfuR4VdqB88SY8UckvLuwelfVzUMF3",
}
IDS_NACIONAL = {
    2021: "1Ac5ZBLm13_UTnyNtT2Mt2U4sBmrTCjqq",
    2022: "1XkdOJjgMK38SuNq73JaP_sQbAebcCsp5",
    2023: "1VP7VoME74WX5kBHKVCfKfchBGIGWyBtk",
    2024: "1Wxczogg9p866djpdRKqJ02qjsGaeocKI",
    2025: "1msgJ1PU1bEbDmPvnz-dAO1RP6SN-QNVw",
}
FONTES = [
    {"universo":"Cerrado", "ano":ano, "caminho":str(PASTA_BASE/f"sicor_cerrado_{ano}.csv"), "drive_id":IDS_CERRADO[ano]}
    for ano in ANOS
]
if EXECUTAR_NACIONAL:
    FONTES += [
        {"universo":"Nacional", "ano":ano, "caminho":str(PASTA_BASE/f"sicor_glebas_wkt_{ano}.csv"), "drive_id":IDS_NACIONAL[ano]}
        for ano in ANOS
    ]
EXECUCAO = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
# Salva na estrutura existente; se essa estrutura não estiver montada, salva no Colab.
PASTA_SAIDA_BASE = PASTA_RELATORIOS if PASTA_RELATORIOS.is_dir() else Path("/content")
SAIDA = PASTA_SAIDA_BASE/"auditoria_geometrias_5_2_3"/EXECUCAO
SAIDA.mkdir(parents=True, exist_ok=True)
CACHE = Path("/content/cache_auditoria_geometrias")
COLUNAS_ID = ["ID_GLEBA","id_gleba2","REF_BACEN","NU_ORDEM","NU_GLEBA"]
print("Resultados serão salvos em:", SAIDA)
for f in FONTES: print(f["universo"],f["ano"],f["caminho"])


Resultados serão salvos em: /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/auditoria_geometrias_5_2_3/20260917_213630_259425
Cerrado 2021 /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/sicor_cerrado_2021.csv
Cerrado 2022 /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/sicor_cerrado_2022.csv
Cerrado 2023 /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/sicor_cerrado_2023.csv
Cerrado 2024 /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/sicor_cerrado_2024.csv
Cerrado 2025 /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/sicor_cerrado_2025.csv
Nacional 2021 /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/sicor_glebas_wkt_2021.csv
Nacional 2022 /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/sicor_glebas_wkt_2022.csv
Nacional 2023 /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/sicor_glebas_wkt_2023.csv
Nacional 2024 /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/sicor_glebas_wkt_2024.c

## 3. Localizar e conferir os arquivos
A conferência verifica a presença da coluna WKT e o separador antes do processamento completo. Nenhuma linha malformada é ignorada. Um erro de leitura interrompe a execução.

In [3]:
import csv, json, zipfile, re, platform, warnings
import pandas as pd
import numpy as np
import shapely
from IPython.display import display, HTML

servico_drive = None
def localizar(fonte):
    global servico_drive
    p=Path(fonte["caminho"])
    if p.is_file(): return p, {"acesso":"caminho_montado","tamanho_bytes":p.stat().st_size}
    if not USAR_DOWNLOAD_POR_ID_SE_NECESSARIO:
        raise FileNotFoundError(f"Arquivo não encontrado: {p}")
    if servico_drive is None:
        from google.colab import auth
        import google.auth
        from googleapiclient.discovery import build
        auth.authenticate_user()
        credenciais,_=google.auth.default()
        servico_drive=build("drive","v3",credentials=credenciais,cache_discovery=False)
    from googleapiclient.http import MediaIoBaseDownload
    meta=servico_drive.files().get(fileId=fonte["drive_id"],fields="id,name,size,modifiedTime,md5Checksum",supportsAllDrives=True).execute()
    if meta["name"] != p.name:
        raise ValueError(f"O arquivo de ID {fonte['drive_id']} mudou de nome: {meta['name']}. Confira antes de continuar.")
    CACHE.mkdir(parents=True,exist_ok=True)
    destino=CACHE/(fonte["drive_id"]+"_"+p.name)
    # O marcador distingue versões com o mesmo tamanho.
    marcador=destino.with_suffix(".origem.json")
    completo=destino.exists() and marcador.exists() and json.loads(marcador.read_text())==meta and destino.stat().st_size==int(meta["size"])
    if not completo:
        tmp=destino.with_suffix(".part")
        req=servico_drive.files().get_media(fileId=fonte["drive_id"],supportsAllDrives=True)
        with tmp.open("wb") as arq:
            dl=MediaIoBaseDownload(arq,req,chunksize=32*1024*1024)
            fim=False
            while not fim:
                status,fim=dl.next_chunk(num_retries=3)
                if status: print(f"Download {p.name}: {100*status.progress():.0f}%")
        if tmp.stat().st_size!=int(meta["size"]): raise RuntimeError("Download incompleto.")
        tmp.replace(destino)
        marcador.write_text(json.dumps(meta),encoding="utf-8")
    return destino,{"acesso":"download_autenticado",**meta}

def conferir_csv(p):
    with p.open("r",encoding=CODIFICACAO,newline="") as arq:
        colunas=next(csv.reader(arq,delimiter=SEPARADOR))
    if len(colunas)!=len(set(colunas)): raise ValueError(f"Cabeçalhos repetidos em {p.name}")
    if COLUNA_WKT not in colunas:
        raise ValueError(f"{p.name}: coluna {COLUNA_WKT!r} ausente. Colunas: {colunas}. Confira separador e codificação.")
    pd.read_csv(p,sep=SEPARADOR,encoding=CODIFICACAO,nrows=5,dtype=str,keep_default_na=False)
    return colunas

ENTRADAS=[]
for fonte in FONTES:
    p,meta=localizar(fonte)
    colunas=conferir_csv(p)
    ENTRADAS.append({**fonte,"caminho_efetivo":str(p),"colunas":colunas,"metadados_acesso":meta})
    print("OK:",fonte["universo"],fonte["ano"],p.name)


OK: Cerrado 2021 sicor_cerrado_2021.csv
OK: Cerrado 2022 sicor_cerrado_2022.csv
OK: Cerrado 2023 sicor_cerrado_2023.csv
OK: Cerrado 2024 sicor_cerrado_2024.csv
OK: Cerrado 2025 sicor_cerrado_2025.csv
OK: Nacional 2021 sicor_glebas_wkt_2021.csv
OK: Nacional 2022 sicor_glebas_wkt_2022.csv
OK: Nacional 2023 sicor_glebas_wkt_2023.csv
OK: Nacional 2024 sicor_glebas_wkt_2024.csv
OK: Nacional 2025 sicor_glebas_wkt_2025.csv


## 4. Auditoria em blocos
WKT não interpretável é contado separadamente de invalidade topológica: um polígono pode ser lido corretamente e ainda apresentar auto-interseção, por exemplo. O parser não corrige o texto.

Documentação: [Shapely — from_wkt](https://shapely.readthedocs.io/en/stable/reference/shapely.from_wkt.html), [is_valid](https://shapely.readthedocs.io/en/stable/reference/shapely.is_valid.html).

Os CSVs detalhados incluem o número sequencial do registro no arquivo (sem contar o cabeçalho), os identificadores disponíveis, o motivo e o WKT problemático. O detalhe pode conter coordenadas dos registros da pesquisa.


In [4]:
CATEGORIAS=["poligono_valido","poligono_invalido","wkt_ausente","wkt_nao_interpretavel","geometria_vazia","tipo_nao_poligonal"]
def classificar(textos):
    textos=np.asarray(textos,dtype=object)
    ausente=pd.Series(textos,dtype="string").fillna("").str.strip().str.casefold().isin(["","null","none","nan","<na>"]).to_numpy()
    # on_invalid='ignore' não repara: retorna None nos erros de decodificação,
    # que são imediatamente contados como wkt_nao_interpretavel.
    geom=shapely.from_wkt(np.where(ausente,None,textos),on_invalid="ignore")
    erro=~ausente & shapely.is_missing(geom)
    vazia=~ausente & ~erro & shapely.is_empty(geom)
    poligono=~ausente & ~erro & ~vazia & np.isin(shapely.get_type_id(geom),[3,6])
    valida=shapely.is_valid(geom)
    status=np.full(len(textos),"tipo_nao_poligonal",dtype=object)
    status[ausente]="wkt_ausente";status[erro]="wkt_nao_interpretavel";status[vazia]="geometria_vazia"
    status[poligono & valida]="poligono_valido";status[poligono & ~valida]="poligono_invalido"
    motivos=np.full(len(textos),"",dtype=object)
    invalidos=status=="poligono_invalido"
    motivos[invalidos]=shapely.is_valid_reason(geom[invalidos])
    motivos[ausente]="WKT ausente";motivos[vazia]="Geometria vazia"
    motivos[status=="tipo_nao_poligonal"]="Tipo diferente de Polygon/MultiPolygon"
    for j in np.flatnonzero(erro):
        try: shapely.from_wkt(textos[j],on_invalid="raise")
        except Exception as exc: motivos[j]=str(exc)
    return status,motivos

def auditar_arquivo(fonte):
    p=Path(fonte["caminho_efetivo"])
    ids=[k for k in COLUNAS_ID if k in fonte["colunas"]]
    contagens={k:0 for k in CATEGORIAS};total=0;motivos_contados={}
    detalhe=SAIDA/f"problemas_{fonte['universo'].lower()}_{fonte['ano']}.csv"
    # Nova execução da célula reescreve só os produtos desta auditoria.
    cabecalho=["universo","ano","arquivo","registro_no_arquivo",*ids,"status","motivo",COLUNA_WKT]
    with detalhe.open("w",encoding="utf-8-sig",newline="") as saida:
        writer=csv.writer(saida,delimiter=";");writer.writerow(cabecalho)
        leitor=pd.read_csv(p,sep=SEPARADOR,encoding=CODIFICACAO,dtype=str,keep_default_na=False,
                           usecols=[COLUNA_WKT,*ids],chunksize=TAMANHO_BLOCO,on_bad_lines="error",skip_blank_lines=False)
        for numero,bloco in enumerate(leitor,1):
            wkt=bloco[COLUNA_WKT].fillna("").to_numpy()
            status,motivos=classificar(wkt)
            for k in CATEGORIAS:contagens[k]+=int(np.count_nonzero(status==k))
            for j in np.flatnonzero(status!="poligono_valido"):
                linha=bloco.iloc[j]
                writer.writerow([fonte["universo"],fonte["ano"],p.name,total+j+1,*[linha[k] for k in ids],status[j],motivos[j],wkt[j]])
                if status[j]=="poligono_invalido":
                    classe=re.sub(r"\[.*$","",str(motivos[j])).strip()
                    motivos_contados[classe]=motivos_contados.get(classe,0)+1
            total+=len(bloco)
            if numero==1 or numero%10==0:print(f"{fonte['universo']} {fonte['ano']} — {total:,} registros; {contagens['poligono_invalido']} polígonos inválidos")
    assert sum(contagens.values())==total
    return {"universo":fonte["universo"],"ano":fonte["ano"],"registros_total":total,**contagens},[
        {"universo":fonte["universo"],"ano":fonte["ano"],"motivo":k,"quantidade":v} for k,v in motivos_contados.items()]

resumos=[];lista_motivos=[]
for fonte in ENTRADAS:
    resumo,motivos=auditar_arquivo(fonte)
    resumos.append(resumo);lista_motivos.extend(motivos)
print("Auditoria concluída. Nenhuma geometria foi corrigida e nenhuma linha foi deduplicada.")


Cerrado 2021 — 50,000 registros; 0 polígonos inválidos
Cerrado 2022 — 50,000 registros; 0 polígonos inválidos
Cerrado 2023 — 50,000 registros; 0 polígonos inválidos
Cerrado 2024 — 50,000 registros; 0 polígonos inválidos
Cerrado 2025 — 50,000 registros; 0 polígonos inválidos
Nacional 2021 — 50,000 registros; 0 polígonos inválidos
Nacional 2021 — 500,000 registros; 0 polígonos inválidos
Nacional 2021 — 980,787 registros; 0 polígonos inválidos
Nacional 2022 — 50,000 registros; 0 polígonos inválidos
Nacional 2022 — 500,000 registros; 2 polígonos inválidos
Nacional 2022 — 1,000,000 registros; 2 polígonos inválidos
Nacional 2023 — 50,000 registros; 0 polígonos inválidos
Nacional 2023 — 500,000 registros; 0 polígonos inválidos
Nacional 2023 — 1,000,000 registros; 1 polígonos inválidos
Nacional 2024 — 50,000 registros; 0 polígonos inválidos
Nacional 2024 — 500,000 registros; 0 polígonos inválidos
Nacional 2024 — 1,000,000 registros; 0 polígonos inválidos
Nacional 2025 — 50,000 registros; 0 pol

## 5. Tabelas anuais e total geral
O total é calculado separadamente para cada universo, sem somar nacional e Cerrado. O percentual total é a razão das somas, não a média dos percentuais anuais. “—” indica denominador zero.

In [5]:
def consolidar(resumos):
    df=pd.DataFrame(resumos)
    saidas=[]
    for universo,g in df.groupby("universo",sort=False):
        g=g.sort_values("ano").copy()
        total={"universo":universo,"ano":"TOTAL",**{k:int(g[k].sum()) for k in ["registros_total",*CATEGORIAS]}}
        g=pd.concat([g,pd.DataFrame([total])],ignore_index=True)
        g["poligonos_avaliados"]=g.poligono_valido+g.poligono_invalido
        g["percentual_invalidos_sobre_poligonos"]=100*g.poligono_invalido/g.poligonos_avaliados.replace(0,np.nan)
        g["percentual_invalidos_sobre_registros"]=100*g.poligono_invalido/g.registros_total.replace(0,np.nan)
        saidas.append(g)
    return pd.concat(saidas,ignore_index=True)

def gravar(df,p):df.to_csv(p,index=False,sep=";",decimal=",",encoding="utf-8-sig",float_format="%.9f")
def pct(x):return "—" if pd.isna(x) else f"{x:.6f}".replace(".",",")
def num(x):return f"{int(x):,}".replace(",",".")
resumo=consolidar(resumos)
gravar(resumo,SAIDA/"resumo_anual_todos_universos.csv")
gravar(pd.DataFrame(lista_motivos,columns=["universo","ano","motivo","quantidade"]),SAIDA/"motivos_invalidade.csv")
for universo,g in resumo.groupby("universo",sort=False):
    gravar(g,SAIDA/f"resumo_anual_{universo.lower()}.csv")
    tabela=g[["ano","registros_total","poligonos_avaliados","poligono_invalido","percentual_invalidos_sobre_poligonos",
              "percentual_invalidos_sobre_registros","wkt_ausente","wkt_nao_interpretavel","geometria_vazia","tipo_nao_poligonal"]].copy()
    for col in tabela:
        if col.startswith("percentual_"):tabela[col]=tabela[col].map(pct)
        elif col!="ano":tabela[col]=tabela[col].map(num)
    tabela.columns=["Ano","Registros","Polígonos avaliados","Polígonos inválidos","Inválidos / polígonos (%)",
                    "Inválidos / registros (%)","WKT ausente","WKT não interpretável","Vazias","Não poligonais"]
    nota="Percentual principal: polígonos inválidos / polígonos não vazios avaliados × 100. Uma linha corresponde a um registro, sem deduplicação. Ano do arquivo de origem."
    pagina=("<!doctype html><html lang='pt-BR'><meta charset='utf-8'><style>body{font:14px Arial;margin:30px}"
            "table{border-collapse:collapse}th,td{padding:8px;border-bottom:1px solid #ccd5c5;text-align:right}"
            "th{background:#e6edde}tr:last-child{font-weight:bold}</style>"
            f"<h2>Qualidade das geometrias — {universo}</h2>"+tabela.to_html(index=False,border=0)+f"<p>{nota}</p></html>")
    (SAIDA/f"tabela_word_{universo.lower()}.html").write_text(pagina,encoding="utf-8")
    print("UNIVERSO:",universo)
    display(HTML(tabela.to_html(index=False,border=0)))


UNIVERSO: Cerrado


Ano,Registros,Polígonos avaliados,Polígonos inválidos,Inválidos / polígonos (%),Inválidos / registros (%),WKT ausente,WKT não interpretável,Vazias,Não poligonais
2021,98.209,98.204,0,"0,000000","0,000000",0,0,0,5
2022,84.771,84.771,0,"0,000000","0,000000",0,0,0,0
2023,93.911,93.911,0,"0,000000","0,000000",0,0,0,0
2024,105.194,105.194,0,"0,000000","0,000000",0,0,0,0
2025,98.594,98.594,0,"0,000000","0,000000",0,0,0,0
TOTAL,480.679,480.674,0,"0,000000","0,000000",0,0,0,5


UNIVERSO: Nacional


Ano,Registros,Polígonos avaliados,Polígonos inválidos,Inválidos / polígonos (%),Inválidos / registros (%),WKT ausente,WKT não interpretável,Vazias,Não poligonais
2021,980.787,980.734,0,"0,000000","0,000000",0,0,0,53
2022,1.062.303,1.062.303,2,"0,000188","0,000188",0,0,0,0
2023,1.109.673,1.109.673,1,"0,000090","0,000090",0,0,0,0
2024,1.166.223,1.166.223,0,"0,000000","0,000000",0,0,0,0
2025,987.163,987.163,0,"0,000000","0,000000",0,0,0,0
TOTAL,5.306.149,5.306.096,3,"0,000057","0,000057",0,0,0,53


## 6. Texto dos resultados e arquivos para download
Use a tabela **Cerrado** para o recorte da pesquisa; use a tabela **Nacional** quando descrever o diagnóstico anterior à seleção territorial. Os dois denominadores não são intercambiáveis. O HTML pode ser aberto no navegador e sua tabela copiada para o Word.

In [6]:
textos=[]
for universo,g in resumo.groupby("universo",sort=False):
    textos += [f"UNIVERSO: {universo}", ""]
    for _,r in g.iterrows():
        periodo="No conjunto de 2021 a 2025" if r.ano=="TOTAL" else f"Em {r.ano}"
        if r.poligonos_avaliados:
            textos.append(f"{periodo}, foram avaliados {num(r.poligonos_avaliados)} registros poligonais não vazios, dos quais {num(r.poligono_invalido)} apresentaram invalidade topológica ({pct(r.percentual_invalidos_sobre_poligonos)}%).")
        else:textos.append(f"{periodo}, não havia polígonos não vazios para calcular o percentual.")
    t=g.iloc[-1]
    textos += [f"Outras ocorrências: WKT ausente = {num(t.wkt_ausente)}; WKT não interpretável = {num(t.wkt_nao_interpretavel)}; geometria vazia = {num(t.geometria_vazia)}; tipo não poligonal = {num(t.tipo_nao_poligonal)}.", ""]
textos += ["Unidade de contagem: registro territorial, sem deduplicação e sem separar partes de MultiPolygon.",
           "A auditoria leu GT_GEOMETRIA sem correção. O código Base01 consultado preserva esse WKT nos arquivos do Cerrado, embora utilize uma geometria temporária corrigida na seleção espacial. Isso não comprova que os arquivos não tenham sofrido modificações posteriores.",
           "O diagnóstico não mede precisão posicional, não identifica sobreposições entre glebas e não comprova que as geometrias foram reparadas com sucesso."]
texto="\n".join(textos)
(SAIDA/"texto_resultados_5_2_3.txt").write_text(texto,encoding="utf-8")
meta={"gerado_em_utc":datetime.now(timezone.utc).isoformat(),"entradas":ENTRADAS,"coluna_wkt":COLUNA_WKT,
      "ano":"Ano de origem do arquivo","tamanho_bloco":TAMANHO_BLOCO,"separador":SEPARADOR,"codificacao":CODIFICACAO,
      "deduplicacao":False,"correcao":False,"recorte_novo":False,"escopo":"Dois universos independentes quando Nacional ativado",
      "versoes":{"python":platform.python_version(),"pandas":pd.__version__,"shapely":shapely.__version__,"geos":shapely.geos_version_string},
      "base01_consultado":"1lB7318tNJWuEJlr4r-EbNpPTCixz-ld8","nota":"Caminhos e IDs conferidos no Drive; WKT original preservado conforme código Base01 consultado."}
(SAIDA/"metadados.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding="utf-8")
zip_final=SAIDA/"Auditoria_Geometrias_Resultados.zip"
with zipfile.ZipFile(zip_final,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SAIDA.iterdir()):
        if p.is_file() and p!=zip_final:z.write(p,arcname=p.name)
print(texto)
print("\nResultados:",SAIDA)
if BAIXAR_ZIP:
    from google.colab import files
    files.download(str(zip_final))


UNIVERSO: Cerrado

Em 2021, foram avaliados 98.204 registros poligonais não vazios, dos quais 0 apresentaram invalidade topológica (0,000000%).
Em 2022, foram avaliados 84.771 registros poligonais não vazios, dos quais 0 apresentaram invalidade topológica (0,000000%).
Em 2023, foram avaliados 93.911 registros poligonais não vazios, dos quais 0 apresentaram invalidade topológica (0,000000%).
Em 2024, foram avaliados 105.194 registros poligonais não vazios, dos quais 0 apresentaram invalidade topológica (0,000000%).
Em 2025, foram avaliados 98.594 registros poligonais não vazios, dos quais 0 apresentaram invalidade topológica (0,000000%).
No conjunto de 2021 a 2025, foram avaliados 480.674 registros poligonais não vazios, dos quais 0 apresentaram invalidade topológica (0,000000%).
Outras ocorrências: WKT ausente = 0; WKT não interpretável = 0; geometria vazia = 0; tipo não poligonal = 5.

UNIVERSO: Nacional

Em 2021, foram avaliados 980.734 registros poligonais não vazios, dos quais 0 ap

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>